# RAGAS Evaluation — OpenAI (from CSV)

Evaluate RAG quality on the testset saved in `testset_openai.csv`.

**Pipeline**:
1. Load questions + reference contexts from `testset_openai.csv`
2. Generate an answer for each question using GPT-4o with the reference contexts as the prompt
3. Run RAGAS metrics on the resulting dataset

**Metrics computed**:
| Metric | What it measures |
|---|---|
| `faithfulness` | Answer is grounded in the retrieved contexts (no hallucination) |
| `answer_relevancy` | Answer directly addresses the question |
| `context_precision` | Retrieved contexts rank relevant chunks higher |
| `context_recall` | Retrieved contexts cover the ground-truth answer |

## 1. Install Dependencies

In [ ]:
%pip install -q \
    "ragas>=0.2.0,<0.3.0" \
    "langchain-openai>=0.1.0" \
    "python-dotenv>=1.0.0"

## 2. Environment Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(".env"), override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Create a .env file in this directory with OPENAI_API_KEY=sk-..."
    )

print(f"OPENAI_API_KEY length: {len(os.environ['OPENAI_API_KEY'])} chars")

## 3. Load Testset from CSV

CSV columns from `ragas-generate-questions-openai.ipynb`:
- `user_input` — the generated question
- `reference_contexts` — JSON-encoded list of source passages
- `reference` — ground-truth answer
- `synthesizer_name` — question type

In [ ]:
import json
import pandas as pd

CSV_PATH = Path("testset_openai.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"{CSV_PATH} not found. "
        "Run ragas-generate-questions-openai.ipynb first to generate it."
    )

df = pd.read_csv(CSV_PATH)
df["reference_contexts"] = df["reference_contexts"].apply(json.loads)

print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Columns: {list(df.columns)}")
print(f"\nQuestion types:\n{df['synthesizer_name'].value_counts().to_string()}\n")
df[["user_input", "reference", "synthesizer_name"]].head(5)

## 4. Initialize OpenAI LLM and Embeddings

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

_llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

evaluator_llm = LangchainLLMWrapper(_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(_embeddings)

print("LLM:", _llm.model_name)
print("Embeddings:", _embeddings.model)
print("Wrappers ready.")

## 5. Generate Answers

For each question, call GPT-4o with the reference contexts injected into the prompt.  
This simulates what your RAG system would return, so RAGAS can measure faithfulness and relevancy.

> To evaluate your *actual* RAG system instead, replace `_generate_answer()` with a call to your API endpoint.

In [ ]:
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    "Bạn là trợ lý HR. Chỉ sử dụng thông tin trong các đoạn văn bản dưới đây "
    "để trả lời câu hỏi. Trả lời bằng tiếng Việt, ngắn gọn và chính xác. "
    "Nếu không tìm thấy câu trả lời trong ngữ cảnh, hãy nói rõ điều đó."
)

def _generate_answer(question: str, contexts: list[str]) -> str:
    context_block = "\n\n---\n\n".join(contexts)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Ngữ cảnh:\n{context_block}\n\nCâu hỏi: {question}",
        },
    ]
    response = _llm.invoke(messages)
    return response.content


tqdm.pandas(desc="Generating answers")
df["response"] = df.progress_apply(
    lambda row: _generate_answer(row["user_input"], row["reference_contexts"]),
    axis=1,
)

print(f"\nGenerated {len(df)} answers.")
df[["user_input", "response"]].head(3)

## 6. Build RAGAS EvaluationDataset

RAGAS 0.2.x uses `EvaluationDataset` built from a list of `SingleTurnSample` objects.  
Each sample needs: `user_input`, `retrieved_contexts`, `response`, `reference`.

In [ ]:
from ragas import EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

samples = [
    SingleTurnSample(
        user_input=row["user_input"],
        retrieved_contexts=row["reference_contexts"],
        response=row["response"],
        reference=row["reference"],
    )
    for _, row in df.iterrows()
]

eval_dataset = EvaluationDataset(samples=samples)
print(f"EvaluationDataset ready: {len(eval_dataset)} samples")

## 7. Run RAGAS Evaluation

**Allow 5–10 minutes** — each metric makes several LLM calls per sample.

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print("\n=== RAGAS Scores ===")
for metric, score in results.items():
    print(f"  {metric:<25} {score:.4f}")

## 8. Inspect Per-Sample Results

In [ ]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 30)

results_df = results.to_pandas()

print(f"Shape: {results_df.shape}")
print(f"Columns: {list(results_df.columns)}\n")
results_df

## 9. Score Summary by Question Type

In [ ]:
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

results_df["synthesizer_name"] = df["synthesizer_name"].values

summary = results_df.groupby("synthesizer_name")[metric_cols].mean().round(4)
overall = results_df[metric_cols].mean().round(4).rename("overall")

print("=== Scores by question type ===")
print(summary.to_string())
print("\n=== Overall ===")
print(overall.to_string())

## 10. Save Results to CSV

In [ ]:
OUTPUT_PATH = Path("evaluation_results_openai.csv")

results_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved {len(results_df)} rows → {OUTPUT_PATH.resolve()}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")